# Day 4 — PySpark: DataFrame Aggregations
**Topics:** groupBy, agg, sum, count, avg, min, max, HAVING equivalent via filter

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'build-de-project-weekend'))
from config.db_config import set_spark_env, get_spark

set_spark_env()
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = get_spark('Day4_Aggregations')

# Sample data — same as SQL notebook
sales_data = [
    ('S001','P001','North','2024-01-05', 10, 29.99),
    ('S002','P002','South','2024-01-07',  5, 49.99),
    ('S003','P001','East', '2024-01-10', 20, 29.99),
    ('S004','P003','West', '2024-01-12', 15, 99.99),
    ('S005','P002','North','2024-01-15',  8, 49.99),
    ('S006','P001','South','2024-02-01', 30, 29.99),
    ('S007','P003','East', '2024-02-03',  2, 99.99),
    ('S008','P004','West', '2024-02-08', 25, 14.99),
    ('S009','P004','North','2024-02-11', 40, 14.99),
    ('S010','P002','East', '2024-02-14', 12, 49.99),
    ('S011','P001','West', '2024-03-01', 18, 29.99),
    ('S012','P003','South','2024-03-05',  7, 99.99),
    ('S013','P004','East', '2024-03-09', 22, 14.99),
    ('S014','P002','West', '2024-03-12',  3, 49.99),
    ('S015','P001','North','2024-03-20', 14, 29.99),
]

schema = StructType([
    StructField('sale_id',    StringType(),  False),
    StructField('product_id', StringType(),  True),
    StructField('region',     StringType(),  True),
    StructField('sale_date',  StringType(),  True),
    StructField('quantity',   IntegerType(), True),
    StructField('unit_price', DoubleType(),  True),
])

df_sales = spark.createDataFrame(sales_data, schema=schema)
df_sales.printSchema()
df_sales.show(5)
print('Total rows:', df_sales.count())

[db_config] PYSPARK_PYTHON        = C:\ApniKaksha\DataEngineer\Practice\python-pyspark-sql-sessions\myenv\Scripts\python.exe
[db_config] JAVA_HOME             = C:\Program Files\Java\jdk-21
[db_config] Spark environment variables set.
[db_config] Spark 4.1.1 ready — app: Day4_Aggregations
[db_config] JDBC jar: C:\ApniKaksha\DataEngineer\Practice\python-pyspark-sql-sessions\myenv\Lib\site-packages\pyspark\jars\postgresql-42.7.3.jar
root
 |-- sale_id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+-------+----------+------+----------+--------+----------+
|sale_id|product_id|region| sale_date|quantity|unit_price|
+-------+----------+------+----------+--------+----------+
|   S001|      P001| North|2024-01-05|      10|     29.99|
|   S002|      P002| South|2024-01-07|       5|     49.99|
|   S003|      P001|

## 1. count — total rows

In [3]:
# count('*') — all rows; count('col') — non-NULL only
df_sales.agg(
    F.count('*').alias('total_rows'),
    F.count('region').alias('non_null_region'),
    F.countDistinct('region').alias('distinct_regions'),
).show()

+----------+---------------+----------------+
|total_rows|non_null_region|distinct_regions|
+----------+---------------+----------------+
|        15|             15|               4|
+----------+---------------+----------------+



## 2. sum — total quantity and revenue

In [4]:
# Global aggregation — no groupBy
df_sales.agg(
    F.sum('quantity').alias('total_units'),
    F.round(
        F.sum(F.col('quantity') * F.col('unit_price')), 2
    ).alias('total_revenue'),
).show()

+-----------+-------------+
|total_units|total_revenue|
+-----------+-------------+
|        231|      7862.69|
+-----------+-------------+



## 3. groupBy + agg — revenue per region

In [5]:
# Option A: withColumn first (more readable)
df_revenue = (
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.sum('quantity').alias('total_units'),
        F.count('*').alias('num_sales'),
    )
    .orderBy('total_revenue', ascending=False)
)
df_revenue.show()

+------+-------------+-----------+---------+
|region|total_revenue|total_units|num_sales|
+------+-------------+-----------+---------+
|  West|      2564.39|         61|        4|
| South|      1849.58|         42|        3|
|  East|      1729.44|         56|        4|
| North|      1719.28|         72|        4|
+------+-------------+-----------+---------+



## 4. min / max / avg per group

In [6]:
df_prices = (
    df_sales
    .groupBy('region')
    .agg(
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
    )
    .orderBy('region')
)
df_prices.show()

+------+---------+---------+---------+
|region|min_price|max_price|avg_price|
+------+---------+---------+---------+
|  East|    14.99|    99.99|    48.74|
| North|    14.99|    49.99|    31.24|
| South|    29.99|    99.99|    59.99|
|  West|    14.99|    99.99|    48.74|
+------+---------+---------+---------+



## 5. HAVING equivalent — filter() after agg()

In [7]:
# SQL HAVING SUM(quantity) > 50
# PySpark: groupBy → agg → filter
df_top = (
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units_sold'))
    .filter(F.col('total_units_sold') > 50)     # ← this IS the HAVING
    .orderBy('total_units_sold', ascending=False)
)
print('Products with >50 units sold:')
df_top.show()

Products with >50 units sold:
+----------+----------------+
|product_id|total_units_sold|
+----------+----------------+
|      P001|              92|
|      P004|              87|
+----------+----------------+



## 6. All five aggregates in one agg() call

In [8]:
df_summary = (
    df_sales
    .groupBy('region')
    .agg(
        F.count('*').alias('num_transactions'),
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
        F.sum('quantity').alias('total_qty'),
    )
    .orderBy('region')
)
df_summary.show()

+------+----------------+---------+---------+---------+---------+
|region|num_transactions|min_price|max_price|avg_price|total_qty|
+------+----------------+---------+---------+---------+---------+
|  East|               4|    14.99|    99.99|    48.74|       56|
| North|               4|    14.99|    49.99|    31.24|       72|
| South|               3|    29.99|    99.99|    59.99|       42|
|  West|               4|    14.99|    99.99|    48.74|       61|
+------+----------------+---------+---------+---------+---------+



## 7. groupBy multiple columns

In [9]:
# Revenue per region + product
df_multi = (
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region', 'product_id')
    .agg(
        F.round(F.sum('revenue'), 2).alias('revenue'),
        F.sum('quantity').alias('units'),
    )
    .orderBy('region', F.col('revenue').desc())
)
df_multi.show()

+------+----------+-------+-----+
|region|product_id|revenue|units|
+------+----------+-------+-----+
|  East|      P002| 599.88|   12|
|  East|      P001|  599.8|   20|
|  East|      P004| 329.78|   22|
|  East|      P003| 199.98|    2|
| North|      P001| 719.76|   24|
| North|      P004|  599.6|   40|
| North|      P002| 399.92|    8|
| South|      P001|  899.7|   30|
| South|      P003| 699.93|    7|
| South|      P002| 249.95|    5|
|  West|      P003|1499.85|   15|
|  West|      P001| 539.82|   18|
|  West|      P004| 374.75|   25|
|  West|      P002| 149.97|    3|
+------+----------+-------+-----+



In [10]:
spark.stop()
print('Spark stopped.')

Spark stopped.
